# JOILang PromptOps: Local Model Benchmark & Render Trace

이 노트북은 JOILang-Server 저장소의 `run_benchmark.py`를 활용하여 실제 로컬 모델 환경에서 blocks 기반 프롬프트 렌더링이 어떻게 동작하는지 한 단계씩 점검하기 위한 디버깅 및 트레이싱 도구입니다.

주요 목적:
1. 로컬 환경 변수 설정 및 단일 행(Row 1) 테스트 파이프라인 정상 구동 확인
2. blocks 렌더링과 legacy_v13_monolith 렌더링 비교
3. Retrieval Pre-mapping 옵션에 따른 동작 확인
4. 생성된 결과 아티팩트(`row_comparison.csv` 등)를 로드하여 검증 결과(DET 점수, 지연 시간, 토큰 수 등) 확인

> 참고: 이 노트북은 전체 모델 비교 실험을 돌리기 위한 것이 아니라, 본격적인 대규모 실험 전 환경과 프롬프트 렌더링 파이프라인을 점검하는 용도입니다.

## Section 1. 환경 변수 및 경로 설정
제공된 가이드에 따라 Conda 가상환경 파이썬과 타겟 GPU 디바이스를 설정합니다.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display

# 작업 디렉토리 기준 경로 설정
NOTEBOOK_DIR = Path.cwd().resolve()
VERSION_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = VERSION_ROOT.parent.parent

# 환경 변수 설정 (로컬 환경에 맞게 수정 가능)
os.environ["JOI_V15_WORKER_PYTHON"] = "/home/mgjeong/miniconda3/envs/l/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

# 실행할 파이썬 스크립트 경로
RUN_BENCHMARK_SCRIPT = VERSION_ROOT / "scripts" / "run_benchmark.py"

# 벤치마크 CLI를 실행할 Python. 보통 현재 Jupyter kernel의 Python을 사용합니다.
# 필요하면 아래 값을 직접 수정하세요. 예: "/home/mgjeong/miniconda3/envs/l/bin/python"
BENCHMARK_PYTHON = sys.executable

print("[Paths]")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"VERSION_ROOT: {VERSION_ROOT}")
print(f"RUN_BENCHMARK_SCRIPT: {RUN_BENCHMARK_SCRIPT}")

print("\n[Environment Variables]")
print(f"WORKER_PYTHON: {os.environ.get('JOI_V15_WORKER_PYTHON')}")
print(f"LOCAL_DEVICE: {os.environ.get('JOI_V15_LOCAL_DEVICE')}")
print(f"BENCHMARK_PYTHON: {BENCHMARK_PYTHON}")

if not RUN_BENCHMARK_SCRIPT.exists():
    print(f"\n[Warning] run_benchmark.py를 찾지 못했습니다: {RUN_BENCHMARK_SCRIPT}")

[Paths]
REPO_ROOT: /root/llm/JOILang-Server
VERSION_ROOT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413
RUN_BENCHMARK_SCRIPT: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py

[Environment Variables]
WORKER_PYTHON: /home/mgjeong/miniconda3/envs/l/bin/python
LOCAL_DEVICE: cuda:0
BENCHMARK_PYTHON: /usr/bin/python3


## Section 2. 유틸리티 함수: 벤치마크 실행 및 최근 실행 결과 불러오기
벤치마크 스크립트 실행 후 `results/model_suite_<timestamp>` 폴더에 생성되는 CSV를 자동으로 읽어오기 위한 헬퍼 함수입니다.

In [2]:
def run_benchmark(extra_args, *, cwd=None):
    """run_benchmark.py를 subprocess로 실행합니다."""
    if not RUN_BENCHMARK_SCRIPT.exists():
        raise FileNotFoundError(f"run_benchmark.py not found: {RUN_BENCHMARK_SCRIPT}")

    cmd = [BENCHMARK_PYTHON, str(RUN_BENCHMARK_SCRIPT), *map(str, extra_args)]
    print("Running command:")
    print(" ".join(cmd))
    print()

    completed = subprocess.run(
        cmd,
        cwd=str(cwd or VERSION_ROOT),
        env=os.environ.copy(),
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"Benchmark failed with exit code {completed.returncode}")
    return completed


def get_latest_result_dir(version_root):
    results_dir = Path(version_root) / "results"
    if not results_dir.exists():
        return None

    subdirs = [
        d for d in results_dir.iterdir()
        if d.is_dir() and d.name.startswith("model_suite_")
    ]
    if not subdirs:
        return None

    return max(subdirs, key=lambda p: p.stat().st_mtime)


def inspect_latest_row_comparison(version_root):
    latest_dir = get_latest_result_dir(version_root)
    if latest_dir is None:
        print("No results directory found.")
        return None

    csv_path = latest_dir / "row_comparison.csv"
    if not csv_path.exists():
        print(f"row_comparison.csv not found in {latest_dir}")
        return None

    print(f"Loading results from: {csv_path}")
    df = pd.read_csv(csv_path)

    # 핵심 확인 지표 필터링
    preferred_cols = [
        "row_no",
        "model_key",
        "det_score",
        "is_pass",
        "failure_reasons",
        "prompt_tokens",
        "latency",
        "generated_code",
    ]
    target_cols = [col for col in preferred_cols if col in df.columns]
    if not target_cols:
        return df
    return df[target_cols]


## Section 3. [Recommended] 초기 Blocks Render 및 로컬 모델 테스트
가장 목적에 부합하는 권장 진입점입니다. `qwen25_coder_7b`로 Row 1에 대해 blocks 렌더링 및 Retrieval Pre-mapping 옵션을 켜서 실행합니다.

체크리스트:
1. Generated JOICode가 비어 있지 않은가?
2. JSON valid인가?
3. DET 점수는 얼마인가?
4. failure reason이 무엇인가?
5. prompt tokens가 얼마인가?
6. LLM latency가 찍히는가?

In [11]:
#MODEL="phi35_mini"

In [10]:
#MODEL="gemma2_9b_it"  #Gemma2 9B-it의 컨텍스트 한계 8192를 훨씬 넘는 20800 tokens 프롬프트

/home/mgjeong/miniconda3/envs/l/bin/python - <<'PY'
> from transformers import AutoConfig
> 
> model_path = "/home/mgjeong/.cache/huggingface/hub/models--google--gemma-2-9b-it/snapshots/11c9b309abf73637e4b6f9a3fa1e92e615547819"
> cfg = AutoConfig.from_pretrained(model_path, local_files_only=True)
> 
> for key in [
>     "max_position_embeddings",
>     "sliding_window",
>     "rope_theta",
>     "model_type",
>     "architectures",
> ]:
>     print(key, "=", getattr(cfg, key, None))
> PY

huggingface-cli login 

pip install   transformers==4.35.*   sentence-transformers==2.2.2   huggingface_hub==0.19.*



In [ ]:
#MODEL="qwen25_coder_7b"
#MODEL="llama31_8b" 
#MODEL="qwen25_coder_14b"

# Debuding 
# rm -f /tmp/joi_local_llm_worker_debug.log
# ~~
# tail -300 /tmp/joi_local_llm_worker_debug.log

In [3]:
print("Running basic block render test on qwen25_coder_14b (Row 1)...")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "phi35_mini",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running basic block render test on qwen25_coder_14b (Row 1)...
Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key phi35_mini --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability

Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/model_suite_20260515_133748
Rows: 1 | Models: phi35_mini
 - phi35_mini (Phi-3.5-mini-instruct): avg_det=0.0000, gt_exact=0/1, pass=0/1, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=1
   top_generation_error: failed to start persistent worker: FileNotFoundError: [Errno 2] No such file or directory: '/home/mgjeong/miniconda3/envs/l/bin/python'
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/version0_15_upd

CompletedProcess(args=['/usr/bin/python3', '/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py', '--suite', 'paper_local5', '--model-key', 'phi35_mini', '--row-no', '1', '--candidate-k', '1', '--repair-attempts', '0', '--det-profile', 'strict', '--prompt-render-mode', 'blocks', '--enable-retrieval-premapping', '--retrieval-topk', '10', '--retrieval-device', 'cpu', '--print-mode', 'compare', '--strict-availability'], returncode=0)

In [14]:
import os
from pathlib import Path

LOCAL_MODELS_DIR = Path("/root/llm/JOILang-Server/local_models").resolve()
os.environ["JOI_V15_LOCAL_MODEL_NAME"] = str(LOCAL_MODELS_DIR / "qwen25_coder_7b")
os.environ["JOI_V15_LOCAL_FILES_ONLY"] = "true"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

Running command:
/usr/bin/python3 /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_7b --row-no 1 --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-device cpu --print-mode compare --strict-availability



Unavailable models for current runtime: qwen25_coder_7b


RuntimeError: Benchmark failed with exit code 1

In [ ]:
# 위 셀 실행 후 생성된 결과를 데이터프레임으로 확인
df_result_1 = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_1 is not None:
    display(df_result_1)

## Section 4. Full Schema Fallback (Retrieval 끄기) 테스트
Retrieval pre-mapping 없이 전체 스키마를 포함시켜 렌더링합니다. 프롬프트가 길어지므로 토큰 수와 latency 증가를 관찰할 수 있습니다.

In [ ]:
print("Running Full Schema Fallback test...")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "blocks",
    "--disable-retrieval-premapping",
    "--print-mode", "compare",
    "--strict-availability",
])

In [ ]:
df_result_full_schema = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_full_schema is not None:
    display(df_result_full_schema)

## Section 5. Legacy Monolith (v13) vs Blocks (v15) 렌더링 모드 비교
`legacy_v13_monolith` 모드를 실행합니다. 이후 Section 3의 blocks 결과와 비교해볼 수 있습니다.

참고: 이 테스트를 위해서는 `version0_13` 디렉토리에 프롬프트 에셋이 존재해야 합니다.

In [ ]:
print("Running Legacy Monolith test...")

prompt_assets_dir = REPO_ROOT / "gpt_mg" / "version0_13"
print(f"prompt_assets_dir: {prompt_assets_dir}")

run_benchmark([
    "--suite", "paper_local5",
    "--model-key", "qwen25_coder_7b",
    "--row-no", "1",
    "--candidate-k", "1",
    "--repair-attempts", "0",
    "--det-profile", "strict",
    "--prompt-render-mode", "legacy_v13_monolith",
    "--prompt-assets-dir", str(prompt_assets_dir),
    "--enable-retrieval-premapping",
    "--retrieval-topk", "10",
    "--retrieval-device", "cpu",
    "--print-mode", "compare",
    "--strict-availability",
])

In [ ]:
df_result_legacy = inspect_latest_row_comparison(VERSION_ROOT)
if df_result_legacy is not None:
    display(df_result_legacy)

## Section 6. 스케일업: 카테고리 및 여러 행 실행 예시
Row 1이 성공적으로 실행된다면, 본격적인 진화를 위해 여러 행을 평가할 수 있습니다.
아래 명령은 `--category 1`과 `--category 2`에 대해 카테고리당 3개씩, 총 6개의 데이터를 평가합니다.

실행 시간이 길어질 수 있으므로 기본적으로 주석 처리되어 있습니다. 필요 시 주석을 해제한 뒤 실행하세요.

In [ ]:
# 필요 시 아래 블록의 주석을 해제하고 실행하세요.
#
# print("Running multi-row category test...")
# run_benchmark([
#     "--suite", "paper_local5",
#     "--model-key", "qwen25_coder_7b",
#     "--category", "1",
#     "--category", "2",
#     "--limit-per-category", "3",
#     "--candidate-k", "1",
#     "--repair-attempts", "0",
#     "--det-profile", "strict",
#     "--prompt-render-mode", "blocks",
#     "--enable-retrieval-premapping",
#     "--retrieval-topk", "10",
#     "--retrieval-device", "cpu",
#     "--print-mode", "summary",
#     "--strict-availability",
# ])

print("주석을 해제하면 multi-row test를 실행할 수 있습니다.")

## Section 7. 결과 아티팩트 디렉토리 점검
가장 최근 실행된 결과 디렉토리 내에 HTML/PDF 리포트 및 각종 요약 CSV가 정상적으로 생성되었는지 확인합니다.

In [ ]:
latest_dir = get_latest_result_dir(VERSION_ROOT)
if latest_dir:
    print(f"Latest Result Directory: {latest_dir.name}\n")
    print("--- Contained Files & Reports ---")
    for item in latest_dir.rglob("*"):
        if item.is_file():
            # 너무 긴 경로는 상대 경로로 잘라서 표기
            rel_path = item.relative_to(latest_dir)
            print(f"- {rel_path}")
else:
    print("No result directory to inspect.")

# [Real Test]

## Local Inference Model 

In [11]:
from pathlib import Path
import subprocess
import os
from datetime import datetime

REPO = Path("/root/llm/JOILang-Server")
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_benchmark.py"

os.environ["JOI_V15_WORKER_PYTHON"] = "/root/llm/je/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

def run_blocks_full(model_key: str):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/block_render_full_{model_key}_{ts}"

    cmd = [
        "python", "-u", str(SCRIPT),
        "--suite", "paper_local5",
        "--model-key", model_key,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "blocks",
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--measure-latency",
        "--measure-vram",
        "--paper-fair-mode",
        "--export-paper-artifacts",
        "--print-mode", "summary",
        "--strict-availability",
        "--output-dir", str(out),
    ]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)
    return out

In [12]:
out_qwen7 = run_blocks_full("qwen25_coder_7b")

python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_7b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --measure-latency --measure-vram --paper-fair-mode --export-paper-artifacts --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_7b_20260515_160027
Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_7b_20260515_160027
Rows: 280 | Models: qwen25_coder_7b
 - qwen25_coder_7b (Qwen2.5-Coder-7B-Instruct): avg_det=79.9240, gt_exact=58/280, pass=156/280, avg_latency=4.8305s, warm_p50=5.0524s, cold_load=14.5697s, avg_prompt_tokens=27952.6, peak_vram=22.0304GB
Suite summary CSV: /root/llm/JOILang-Server/gpt_mg/ve

In [10]:
out_qwen7

PosixPath('/root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_7b_20260515_145402')

In [13]:
out_llama = run_blocks_full("llama31_8b")

python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key llama31_8b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --measure-latency --measure-vram --paper-fair-mode --export-paper-artifacts --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_llama31_8b_20260515_163211
Output directory: /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_llama31_8b_20260515_163211
Rows: 280 | Models: llama31_8b
 - llama31_8b (Llama-3.1-8B-Instruct): avg_det=0.0000, gt_exact=0/280, pass=0/280, avg_latency=0.0000s, warm_p50=0.0000s, cold_load=0.0000s, avg_prompt_tokens=0.0, peak_vram=0.0000GB, gen_errors=280
   top_generation_error: We couldn't connect to 'https://huggingface.co' to 

In [ ]:
out_qwen14 = run_blocks_full("qwen25_coder_14b")

python -u /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/scripts/run_benchmark.py --suite paper_local5 --model-key qwen25_coder_14b --candidate-k 1 --repair-attempts 0 --det-profile strict --prompt-render-mode blocks --enable-retrieval-premapping --retrieval-topk 10 --retrieval-mode hybrid --retrieval-device cpu --measure-latency --measure-vram --paper-fair-mode --export-paper-artifacts --print-mode summary --strict-availability --output-dir /root/llm/JOILang-Server/gpt_mg/version0_15_update20260413/results/block_render_full_qwen25_coder_14b_20260515_165037


## 2. Cloud GPT-4.1-mini blocks render 실행

cloud reference도 같은 방식으로 저장해두고 비교하려면, suite만 paper_with_cloud_ref로 바꾸고 model-key를 gpt41_mini로 지정하면 됩니다. run_benchmark.py는 실제로 run_model_suite_benchmark.py의 entrypoint를 호출하고, 그 안에 paper_with_cloud_ref = paper_local5 + GPT-4.1-mini suite가 정의

In [ ]:
from pathlib import Path
import subprocess
import os
from datetime import datetime
import pandas as pd
import json

REPO = Path("/root/llm/JOILang-Server")
SCRIPT = REPO / "gpt_mg/version0_15_update20260413/scripts/run_benchmark.py"

os.environ["JOI_V15_WORKER_PYTHON"] = "/home/mgjeong/miniconda3/envs/l/bin/python"
os.environ["JOI_V15_LOCAL_DEVICE"] = "cuda:0"

# OpenAI-compatible endpoint가 이미 떠 있어야 합니다.
# 예: export JOI_V15_OPENAI_ENDPOINT="http://127.0.0.1:8000/v1/chat/completions"
OPENAI_ENDPOINT = os.environ.get("JOI_V15_OPENAI_ENDPOINT", "http://127.0.0.1:8000/v1/chat/completions")

In [ ]:
def run_cloud_blocks_full(
    row_no=None,
    categories=None,
    limit_per_category=None,
    output_prefix="cloud_blocks_full",
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/{output_prefix}_gpt41_mini_{ts}"

    cmd = [
        "python", "-u", str(SCRIPT),
        "--suite", "paper_with_cloud_ref",
        "--model-key", "gpt41_mini",
        "--llm-mode", "openai",
        "--llm-endpoint", OPENAI_ENDPOINT,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "blocks",
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--print-mode", "summary",
        "--output-dir", str(out),
    ]

    if row_no is not None:
        cmd += ["--row-no", str(row_no)]

    if categories:
        for cat in categories:
            cmd += ["--category", str(cat)]

    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

    return out

In [ ]:
# Row 1만 cloud blocks render로 확인
out_cloud_row1 = run_cloud_blocks_full(row_no=1, output_prefix="cloud_blocks_row1")

In [ ]:
# 전체 dataset cloud blocks render
out_cloud_full = run_cloud_blocks_full(output_prefix="cloud_blocks_full")

In [ ]:
# category별 일부만 먼저 확인
out_cloud_cat12 = run_cloud_blocks_full(
    categories=[1, 2],
    limit_per_category=5,
    output_prefix="cloud_blocks_cat12"
)

### 3. 결과확인

In [ ]:
import pandas as pd
from pathlib import Path

def inspect_result(out_dir):
    out_dir = Path(out_dir)

    print("OUT:", out_dir)

    files = [
        "suite_summary.csv",
        "main_model_comparison.csv",
        "tradeoff_summary.csv",
        "latency_summary.csv",
        "vram_summary.csv",
        "tokenizer_summary.csv",
        "paper_metrics_summary.json",
        "row_comparison.csv",
        "failure_reason_summary.csv",
        "category_summary.csv",
    ]

    for f in files:
        p = out_dir / f
        print(f, "OK" if p.exists() else "MISSING")

    summary = out_dir / "suite_summary.csv"
    if summary.exists():
        display(pd.read_csv(summary))

    main = out_dir / "main_model_comparison.csv"
    if main.exists():
        display(pd.read_csv(main))

    failure = out_dir / "failure_reason_summary.csv"
    if failure.exists():
        display(pd.read_csv(failure).head(20))

    category = out_dir / "category_summary.csv"
    if category.exists():
        display(pd.read_csv(category))

In [ ]:
# Cloud legacy monolith도 비교하고 싶을 때
def run_cloud_legacy_full(
    row_no=None,
    categories=None,
    limit_per_category=None,
    output_prefix="cloud_legacy_full",
):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = REPO / f"gpt_mg/version0_15_update20260413/results/{output_prefix}_gpt41_mini_{ts}"

    prompt_assets = REPO / "gpt_mg/version0_13"

    cmd = [
        "python", "-u", str(SCRIPT),
        "--suite", "paper_with_cloud_ref",
        "--model-key", "gpt41_mini",
        "--llm-mode", "openai",
        "--llm-endpoint", OPENAI_ENDPOINT,
        "--candidate-k", "1",
        "--repair-attempts", "0",
        "--det-profile", "strict",
        "--prompt-render-mode", "legacy_v13_monolith",
        "--prompt-assets-dir", str(prompt_assets),
        "--enable-retrieval-premapping",
        "--retrieval-topk", "10",
        "--retrieval-mode", "hybrid",
        "--retrieval-device", "cpu",
        "--print-mode", "summary",
        "--output-dir", str(out),
    ]

    if row_no is not None:
        cmd += ["--row-no", str(row_no)]

    if categories:
        for cat in categories:
            cmd += ["--category", str(cat)]

    if limit_per_category is not None:
        cmd += ["--limit-per-category", str(limit_per_category)]

    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, check=True)

    return out

In [ ]:
out_cloud_legacy_row1 = run_cloud_legacy_full(row_no=1, output_prefix="cloud_blocks_full")

In [ ]:
# 결과를 Jupyter에서 바로 읽기

def load_result_tables(out_dir):
    out_dir = Path(out_dir)

    files = {
        "suite_summary": out_dir / "suite_summary.csv",
        "row_comparison": out_dir / "row_comparison.csv",
        "failure_reason_summary": out_dir / "failure_reason_summary.csv",
        "category_summary": out_dir / "category_summary.csv",
        "main_model_comparison": out_dir / "main_model_comparison.csv",
        "tradeoff_summary": out_dir / "tradeoff_summary.csv",
        "paper_metrics_summary": out_dir / "paper_metrics_summary.json",
    }

    loaded = {}
    for name, path in files.items():
        if path.exists():
            if path.suffix == ".csv":
                loaded[name] = pd.read_csv(path)
            elif path.suffix == ".json":
                loaded[name] = json.loads(path.read_text(encoding="utf-8"))
        else:
            print(f"[MISSING] {name}: {path}")

    return loaded

In [ ]:
cloud_tables = load_result_tables(out_cloud_full)

display(cloud_tables["suite_summary"])
display(cloud_tables["category_summary"])
display(cloud_tables["failure_reason_summary"].head(20))

In [ ]:
# Cloud blocks vs cloud legacy 비교
out_cloud_blocks_row1 = run_cloud_blocks_full(row_no=1, output_prefix="cloud_blocks_row1")
out_cloud_legacy_row1 = run_cloud_legacy_full(row_no=1, output_prefix="cloud_legacy_row1")

cmp = pd.DataFrame([
    summarize_run(out_cloud_blocks_row1, "gpt41_blocks_row1"),
    summarize_run(out_cloud_legacy_row1, "gpt41_legacy_row1"),
])
display(cmp)

### Local 결과와 Cloud 결과 비교

In [ ]:
out_qwen7 = run_blocks_full("qwen25_coder_7b")
out_cloud_full = run_cloud_blocks_full()

In [ ]:
def summarize_run(out_dir, label):
    tables = load_result_tables(out_dir)
    suite = tables.get("suite_summary")
    if suite is None:
        return None

    row = suite.iloc[0].to_dict()
    row["run_label"] = label
    row["out_dir"] = str(out_dir)
    return row

summary_rows = []
summary_rows.append(summarize_run(out_qwen7, "qwen7_blocks"))
summary_rows.append(summarize_run(out_cloud_full, "gpt41_blocks"))

summary_df = pd.DataFrame([r for r in summary_rows if r is not None])
display(summary_df)

In [ ]:
# 1) Cloud blocks row 1 smoke
out_cloud_row1 = run_cloud_blocks_full(row_no=1, output_prefix="cloud_blocks_row1")

# 2) Local qwen7 row 1 smoke
out_qwen7_row1 = run_blocks_full("qwen25_coder_7b", row_no=1)

# 3) Cloud blocks category 일부
out_cloud_cat12 = run_cloud_blocks_full(categories=[1, 2], limit_per_category=5)

# 4) Local qwen7 category 일부
out_qwen7_cat12 = run_blocks_full("qwen25_coder_7b", categories=[1, 2], limit_per_category=5)

# 5) 문제 없으면 cloud full
out_cloud_full = run_cloud_blocks_full(output_prefix="cloud_blocks_full")

# 6) 문제 없으면 local full
out_qwen7_full = run_blocks_full("qwen25_coder_7b", output_prefix="qwen7_blocks_full")